# Notebook 02: Multi-Class Segmentation

**Module:** 07 Segmentation  
**Duration:** ~2 hrs

---

## Learning Objectives

1. Distinguish multi-class from binary
2. Use softmax output per pixel
3. Compute mIoU across classes

## Multi-Class Segmentation

Each pixel belongs to **exactly one** of $K$ classes (mutually exclusive).

**Output:** $(K, H, W)$ logits → softmax → argmax → $(H, W)$ label map.

**Loss:** Cross-entropy per pixel.

**Metric:** mIoU (mean IoU across classes).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

plt.rcParams['figure.figsize'] = (8, 5)
torch.manual_seed(42)
rng = np.random.default_rng(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
def pixel_accuracy(pred, target):
    return (pred == target).float().mean().item()

def iou_score(pred, target, n_classes=None, smooth=1e-7):
    """pred, target: (N,H,W) long tensors"""
    if n_classes is None:
        n_classes = int(max(pred.max(), target.max())) + 1
    ious = []
    for c in range(n_classes):
        p = (pred == c)
        t = (target == c)
        inter = (p & t).sum().float()
        union = (p | t).sum().float()
        if union > 0:
            ious.append(((inter + smooth) / (union + smooth)).item())
    return float(np.mean(ious)) if ious else 0.0

def dice_score(pred, target, smooth=1e-7):
    """Binary pred, target: (N,1,H,W) float"""
    inter = (pred * target).sum(dim=(2,3))
    return ((2*inter + smooth) / (pred.sum(dim=(2,3)) + target.sum(dim=(2,3)) + smooth)).mean().item()

In [ ]:
# 3-class synthetic: background, road, building
N, H, W, K = 16, 32, 32, 3
target = torch.randint(0, K, (N, H, W))
logits = torch.randn(N, K, H, W)
pred = logits.argmax(dim=1)

print(f'Pixel accuracy: {pixel_accuracy(pred, target):.4f}')
print(f'mIoU: {iou_score(pred, target, K):.4f}')

## GeoSpatial: Multi-class land cover (water, vegetation, urban, bare soil) from satellite imagery.

---

## Summary

Multi-class: one label per pixel, softmax + cross-entropy.

**Next:** [03_Multi_Label_Segmentation.ipynb](03_Multi_Label_Segmentation.ipynb)